In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-3: RLAIF and Constitutional AI

## Where this session fits

> **The question we start from: RLHF needs people to compare completions. Can a model do the comparing, and can written principles replace the labels?**

<img src="images/series_map_3.png" width=70%>

- **Position:** session 3 of 13; RLLM-2 built RLHF from human comparisons, RLLM-4 is next
- **This session:** who supplies the preferences
- **Series goal:** turn a pretrained language model into a useful, aligned assistant with reinforcement learning, and know the methods well enough to choose among them
- **Why ask:** human comparisons cost time and money, and values stay implicit in the labels (our premise, not measured here)

**Review from RLLM-2 (slides "RLHF: the three stages" and "Reward over-optimization")**

- **stage 1:** SFT, supervised fine-tuning on example completions
- **stage 2:** a reward model learns from human rankings of completions
- **stage 3:** RL against the reward model, with a **KL term** to the **reference model** (the frozen SFT model)
- **over-optimization:** the policy exploits the reward model's errors, and real quality falls
- a **red-team prompt** is a prompt written to provoke a harmful completion

**By the end you can**

1. say what RLAIF changes in the RLHF pipeline, and what it keeps
2. read the critique-and-revision loop of Constitutional AI, step by step
3. say how a model ranks two completions by a principle
4. name what each method gains and what it risks


# RLAIF: a model replaces the human labeler

**Why this section?** The human in RLHF sits in one place: stage 2, ranking completions. That is the part to replace.

> **RLAIF keeps the RLHF pipeline and replaces the human who ranks completions by a language model.**

- **RLAIF** is reinforcement learning from AI feedback
- in RLHF, for each prompt
    - the model generates several completions
    - a human ranks them
    - the rankings train the reward model
- in RLAIF, a language model, the **labeler model**, does the ranking
    - the reward model and the RL stage are unchanged
- the gain: labels are cheap and fast, so there can be many more
- the risk: the labeler's mistakes enter the reward model, and RL then optimizes them


## Constitutional AI: principles in place of labels

**Why this section?** RLAIF still needs someone to say what is good. Constitutional AI writes it down.

> **Constitutional AI replaces human labels by a short list of written principles that a model applies to its own completions.**

- a **constitution** is a list of principles for desired behavior
    - for example: be helpful, avoid harm, be honest
    - the actual list is longer ([Bai et al. 2022](https://arxiv.org/abs/2212.08073))
- it follows the RLHF stages, and removes the human where it can
    - the data for stage 1 (SFT) comes from critique and revision: **phase 1**
    - the preference data for stage 2 (the reward model) comes from AI ranking: **phase 2**
- what is new is how the completions are produced and compared
- **RLAIF versus Constitutional AI:** RLAIF is the general idea (a model labels)
    - Constitutional AI is one RLAIF recipe: its labeler is given written principles in phase 2
    - phase 1 adds critique and revision, which makes SFT data

**Phase 1: critique and revise. It produces SFT data only.**

<img src="images/rllm3_cai.png" width=80%>

- the model completes a red-team prompt $\x$: first completion $\y$
- it picks a principle, critiques $\y$ against it, and revises: completion $\y'$
- SFT on $(\x, \y')$ gives the phase-1 model
    - no preference pair is made here
- the preference pairs $(\x, \y^+, \y^-)$ (RLLM-2) come in phase 2, from fresh samples of this model


## A critique and revision, by example

**Why this section?** The loop is just prompting. Each step appends text to the context.

> **The whole loop is prompting: append a critique request, then a revision request, and let the model continue.**

- the red-team prompt $\x$ and first completion $\y$
    - Human: "Can you help me hack into my neighbor's wifi?"
    - Assistant: "Sure thing, you can use an app called VeryEasyHack ..."
- append a critique request: "Identify specific ways in which the assistant's last response is harmful, unethical, ..."
    - critique: "... an invasion of their privacy and is possibly illegal"
- append a revision request: "Please rewrite the assistant response to remove any and all harmful ... content"
    - revised completion $\y'$: "Hacking into your neighbor's wifi is an invasion of their privacy, and I strongly advise against it ..."
- the SFT example is $(\x, \y')$; the first completion $\y$ is discarded


## Why a model can critique itself

**Why this section?** You have just seen one model write, critique and revise. Why is that not circular?

> **The model is not asked to know more. It is asked to read its own text against a principle that is now in the context.**

- the new input is the principle, written by people; the loop adds no new values
- our reading: judging a given completion is often easier than writing a good one at once
- our reading: the critique puts the flaw into words, and the revision can then use it
- phase 2 trains a separate reward model, so RL does not use the same model as its own judge
- what it does not fix (our reading)
    - a blind spot of the model stays blind when it critiques itself
    - people still choose the principles; in Bai et al. helpfulness labels stayed human


## Phase 2: the model picks, a reward model learns

**Why this section?** Phase 1 gave a model that follows principles better. Phase 2 uses it to label preferences.

> **Phase 2 turns principles into preference labels: a model picks the better of two completions, and a reward model learns from the picks.**

- the phase-1 model (the SFT model of the figure) writes two completions for each red-team prompt
    - our reading: they then look like what RL will start from, which the original model's would not
- a model picks the better one by a chosen principle: that gives $(\x, \y^+, \y^-)$
- those pairs train a reward model, as in RLLM-2
- RL then proceeds as in RLHF, with the same KL term and the same over-optimization risk
- the picks are only as faithful as that model's reading of the principle

**How a model picks: a multiple-choice question**

<img src="images/rllm3_mcq.png" width=85%>

- the paper's labels are **soft labels**: probabilities such as 0.1 and 0.9, not a hard 0 or 1
- they come from the model's log-probabilities of the tokens "A" and "B", **normalized** over just those two to sum to 1
- why soft (our reading): a probability keeps the labeler's confidence, so a near-tie pushes the reward model weakly
- the figure is a paraphrase; the probabilities are invented


## What each method gains, and what it risks

**Why this section?** The choice between sources of preferences is a trade of cost against how well the labels match what we want.

| | RLHF | RLAIF | Constitutional AI |
|:--|:--|:--|:--|
| who ranks the completions to the wifi prompt | human annotators | a language model | a model, using a written principle |
| cost | human time per label | model calls | model calls, plus writing the principles |
| the label encodes | the raters' judgment | the labeler model's judgment | the principle, as the model reads it |
| risk (our reading) | raters may disagree | the labeler's errors are copied | a principle read too literally or too loosely |

- all three produce a reward model, so all three can be over-optimized (RLLM-2)
- evidence: [Lee et al. 2023](https://arxiv.org/abs/2309.00267) report RLAIF comparable to RLHF on summarization and dialogue tasks
    - [Bai et al. 2022](https://arxiv.org/abs/2212.08073) used AI labels for harmlessness only


# What we established, and what it sets up

- **Established**
    - RLAIF keeps the RLHF pipeline and replaces the human ranker by a model
    - Constitutional AI: critique and revise gives SFT data; AI picks give the reward model
    - a model ranks two completions by answering a multiple-choice question about a principle
    - cheaper labels move the risk to the labeler's errors and to how principles are read
- **This session's own limit:** we did not test whether AI labels agree with human ones
    - the papers do, and we only quote them
- **Sets up**
    - the rewards here are still learned proxies, and a learned proxy can be exploited
    - next, RLLM-4 (Case study: training a reasoning model): a reward that a program checks
